# Filter ethene dynamic CASSCF data

Discard a geometry when its C=C bond is longer than 2.0 Å or any of its four C--H bonds is longer than 1.5 Å. Complete extended-XYZ records, including valid CASSCF labels, are retained unchanged.

In [1]:
from pathlib import Path

import numpy as np

INPUT_XYZ = Path("../data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF.xyz")
OUTPUT_XYZ = Path("../data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF_filtered.xyz")
MAX_CC_BOND = 2.0  # Å
MAX_CH_BOND = 1.5  # Å

# Atom order in this dataset is C, C, H, H, H, H.
CC_PAIR = (0, 1)
CH_PAIRS = ((0, 2), (0, 5), (1, 3), (1, 4))


def distance(positions, atom_pair):
    first, second = atom_pair
    return float(np.linalg.norm(positions[first] - positions[second]))


stats = {"total": 0, "kept": 0, "discarded_cc": 0, "discarded_ch": 0, "discarded_both": 0}
with INPUT_XYZ.open() as source, OUTPUT_XYZ.open("w") as destination:
    while atom_count_line := source.readline():
        atom_count = int(atom_count_line)
        comment_line = source.readline()
        atom_lines = [source.readline() for _ in range(atom_count)]
        if not comment_line or len(atom_lines) != atom_count or any(not line for line in atom_lines):
            raise ValueError(f"Incomplete frame after frame {stats['total']}")

        atom_fields = [line.split() for line in atom_lines]
        symbols = [fields[0] for fields in atom_fields]
        if atom_count != 6 or symbols != ["C", "C", "H", "H", "H", "H"]:
            raise ValueError(f"Unexpected atom ordering in frame {stats['total']}: {symbols}")
        positions = np.array([[float(value) for value in fields[1:4]] for fields in atom_fields])

        cc_bond = distance(positions, CC_PAIR)
        longest_ch_bond = max(distance(positions, pair) for pair in CH_PAIRS)
        exceeds_cc = cc_bond > MAX_CC_BOND
        exceeds_ch = longest_ch_bond > MAX_CH_BOND

        stats["total"] += 1
        if exceeds_cc:
            stats["discarded_cc"] += 1
        if exceeds_ch:
            stats["discarded_ch"] += 1
        if exceeds_cc and exceeds_ch:
            stats["discarded_both"] += 1
        if exceeds_cc or exceeds_ch:
            continue

        destination.write(atom_count_line)
        destination.write(comment_line)
        destination.writelines(atom_lines)
        stats["kept"] += 1

stats["discarded"] = stats["total"] - stats["kept"]
print(f"Wrote {stats['kept']} retained geometries to {OUTPUT_XYZ.resolve()}")
print(stats)


Wrote 35003 retained geometries to /home/lim_yt/X-MACE-sampling/data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF_filtered.xyz
{'total': 62031, 'kept': 35003, 'discarded_cc': 8642, 'discarded_ch': 18701, 'discarded_both': 315, 'discarded': 27028}
